[← 03 - Aggregation](<03-aggregation.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# 04 - JOINs: INNER vs LEFT, and why grain matters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/04-joins.ipynb)

`03 - Aggregation` turned rows into counts and totals, and this notebook checks which rows actually reach those calculations once a second table is involved. A join combines rows from two tables using a condition you write, and depending on that condition, and on which unmatched rows you decide to keep, the same two tables can hand you three rows, five rows, or a total that's quietly wrong in a way nothing will flag for you.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** combine two tables with `INNER JOIN` and `LEFT JOIN`, say in one sentence what a single result row represents (its grain), spot the ways a join quietly corrupts a total or drops rows you meant to keep, and run the same checks on a real 19,820-customer database.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 4 of the All About Data & More curriculum](LESSON_4_URL), which walks through the reasoning in more depth.

## 0. Set up

Run the cell below once each time you open the notebook. After it, any cell that starts with `%%sql` runs SQLite against a small lesson database that lives in memory, so there's nothing to install, and any cell that starts with `%%tsql` runs against SQL Server and AdventureWorks (only section 10 uses that).

In [1]:
# Run this cell first. It loads the small helper that lets the cells below run SQL,
# downloading it from the repo if you opened this notebook on its own (in Colab, say).
import os, sys, urllib.request
here = next((d for d in ('.', '..') if os.path.exists(os.path.join(d, 'sql_tutorial.py'))), None)
if here is None:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py', 'sql_tutorial.py')
    here = '.'
sys.path.insert(0, here)
%load_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. The two lesson tables

These customers and amounts are invented for this lesson, and they're the same tables the newsletter lesson uses, so every result below should match the article exactly. Run this setup once; if you ever want a clean start, run `%sql_reset` in a new cell and then run this setup again.

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80);

4 rows inserted
3 rows inserted


Before joining anything, look at each table on its own. It's a habit worth keeping on real data too, because most join surprises are visible in the inputs if you look before you combine them.

In [3]:
%%sql
SELECT
    customer_id,
    customer_name,
    credit_limit
FROM customers;

customer_id,customer_name,credit_limit
1,Ada,100
2,Bola,200
3,Chidi,300
4,Dami,200


In [4]:
%%sql
SELECT
    order_id,
    customer_id,
    amount
FROM orders;

order_id,customer_id,amount
101,1,40
102,1,60
103,2,80


The **grain** of a table means what one row represents: in `customers` it's one customer, and in `orders` it's one order. Ada has two orders, Bola has one, and Chidi and Dami have none.

Each **primary key** identifies a row uniquely, and the `customer_id` in `orders` records which customer placed it; the join compares those IDs rather than names, since two different people can share a name. This small setup doesn't declare a foreign key constraint, so nothing stops an order from pointing at a customer who doesn't exist (`08a` comes back to that).

## 2. What a join actually does

A join tests rows from one table against rows from the other, checking each pair against the condition you write after `ON`. When a pair satisfies the condition, the two rows combine into one result row with columns from both tables sitting side by side. What happens to a row that never finds a partner depends on which kind of join you wrote, and that single decision is most of what this notebook is about.

<p align="center">
  <img src="graphics/04_join_rows.png" width="620" alt="INNER JOIN keeps three matched rows; LEFT JOIN keeps the same three plus Chidi and Dami with NULL order columns, five rows in all">
</p>

## 3. INNER JOIN returns matching pairs

`INNER JOIN` throws away any row, from either table, that never finds a match. That's the "inner" in the name: only the overlap survives.

In [5]:
%%sql
SELECT
    c.customer_id,
    c.customer_name,
    o.order_id,
    o.amount
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
ORDER BY
    c.customer_id,
    o.order_id;

customer_id,customer_name,order_id,amount
1,Ada,101,40
1,Ada,102,60
2,Bola,103,80


`c` and `o` are **aliases**, short names for the tables inside this one query, and `c.customer_id` says which table supplies the column while `ON` supplies the matching condition.

Ada's customer row matches two order rows, so it contributes two result rows, and Bola's single match contributes one. Chidi and Dami have no matching order, so neither appears, and an order that pointed at a missing customer would be absent from this result too. That's exactly right for an order list, and exactly wrong the moment the question needs customers who haven't ordered.

## 4. Forget the ON clause and get every possible pairing

`ON` is what makes this a join instead of a listing of every combination. Leave it out, or write two table names separated by a comma, and SQLite runs the query anyway:

In [6]:
%%sql
SELECT
    c.customer_name,
    o.order_id
FROM customers AS c, orders AS o
ORDER BY
    c.customer_id,
    o.order_id;

customer_name,order_id
Ada,101
Ada,102
Ada,103
Bola,101
Bola,102
Bola,103
Chidi,101
Chidi,102
Chidi,103
Dami,101


Four customers times three orders is twelve rows: every possible pairing, not the two orders Ada placed and the one Bola placed. Ada is now sitting next to order 103, which belongs to Bola, because nothing in the query compared `customer_id` with `customer_id`.

This is a **cross join**, or cartesian product, and it's a real result rather than an error. On four customers and three orders it's obviously wrong, but on tables with thousands of rows on each side the row count explodes into the millions, the query gets slow or times out, and if it does finish, most of its pairings are nonsense that passed every syntax check. Always write an explicit `ON` condition, even one you think is implied by the table names.

## 5. LEFT JOIN also keeps customers without orders

The left table is the one written before `LEFT JOIN`, here `customers`, and every one of its rows survives whether or not it finds a partner:

In [7]:
%%sql
SELECT
    c.customer_id,
    c.customer_name,
    o.order_id,
    o.amount
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
ORDER BY
    c.customer_id,
    o.order_id;

customer_id,customer_name,order_id,amount
1,Ada,101,40
1,Ada,102,60
2,Bola,103,80
3,Chidi,NULL,NULL
4,Dami,NULL,NULL


The first three rows are unchanged, and each unmatched customer gets one row with `NULL` in the order columns. `NULL` marks a missing value, here meaning "there was no matching order", and it's a different thing from zero or an empty string.

Notice that keeping every customer hasn't produced one row per customer, since Ada still appears twice. The grain of this result is one row per matched order, plus one row for each customer without a match, and that description only holds because `customer_id` is unique in `customers`. A join returns every matching pair, so if a key appeared twice on the left and three times on the right, it would contribute six rows; real source tables are less tidy than this one, so check a key is actually unique before you assume another dataset behaves like this one.

SQL also has `RIGHT JOIN`, which keeps every row from the table on the right instead, and `FULL JOIN`, which keeps unmatched rows from both sides:

<p align="center">
  <img src="graphics/04_join_types_overview.png" width="640" alt="What each join type keeps: INNER keeps only matched rows, LEFT adds left-only rows, RIGHT adds right-only rows, FULL keeps all three groups">
</p>

> **Portability:** SQL Server and PostgreSQL support all four. SQLite only added `RIGHT JOIN` and `FULL JOIN` in [version 3.39.0, released in 2022](https://sqlite.org/releaselog/3_39_0.html), and MySQL still has no `FULL JOIN` at all (see its [JOIN reference](https://dev.mysql.com/doc/refman/8.4/en/join.html)), so check what your engine supports before copying syntax from a tutorial written for a different one. In practice most people rewrite a `RIGHT JOIN` as a `LEFT JOIN` with the tables swapped, since reading left to right is easier.

## 6. Check grain before adding numbers

Suppose your manager wants the total credit extended to customers, and you add it up after that left join:

In [8]:
%%sql
SELECT SUM(c.credit_limit) AS joined_credit
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id;

joined_credit
900


The customer table only contains 800 in credit limits; scroll up and add them yourself. The joined result counts Ada's 100 twice because she has two orders, so nobody raised her limit, the query just repeated the value once for every order she placed.

<p align="center">
  <img src="graphics/04_repeated_credit.png" width="460" alt="Ada's single customer row matches two orders, so her credit limit of 100 appears twice and sums to 200, while at customer grain it is still 100">
</p>

For a total across customers, read it at customer grain:

In [9]:
%%sql
SELECT SUM(credit_limit) AS customer_credit
FROM customers;

customer_credit
800


A tempting "fix" is `SUM(DISTINCT ...)`, which adds each distinct value once. Run it and see why it isn't one:

In [10]:
%%sql
SELECT SUM(DISTINCT c.credit_limit) AS distinct_credit
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id;

distinct_credit
600


600, not 800. Bola and Dami both have a legitimate limit of 200, and `DISTINCT` collapses their two equal amounts into one, even though they belong to different customers. It removes repeated *values*, while the problem was repeated *rows*, and those are not the same thing.

Order amounts don't have this problem in this join, since each order matches exactly one customer, which is the general rule worth keeping: whether a number is safe to add after a join depends on which table it came from and how many times that table's rows now appear.

## 7. Count orders, including zero

Now combine the join with lesson 3's grouping to get one row per customer:

In [11]:
%%sql
SELECT
    c.customer_id,
    c.customer_name,
    COUNT(*) AS joined_rows,
    COUNT(o.order_id) AS order_count
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY
    c.customer_id;

customer_id,customer_name,joined_rows,order_count
1,Ada,2,2
2,Bola,1,1
3,Chidi,1,0
4,Dami,1,0


Chidi has one joined row but zero orders. `COUNT(*)` counts rows, including the row the left join kept for him, while `COUNT(o.order_id)` skips `NULL`s, so it gives the zero we actually want. The two counts agree for Ada and Bola, which is exactly why this mistake survives testing on customers who do have orders.

Count a column that can't be `NULL` in a real matched order, like its primary key, since counting an optional field such as a delivery note would also miss orders that simply have no note, and group by the customer ID as well as the name so two people called Ada stay separate.

## 8. Common mistakes

### Filtering away the customers you kept

Ask for every customer, with any orders of at least 60. This looks reasonable:

In [12]:
%%sql
SELECT
    c.customer_id,
    c.customer_name,
    o.order_id,
    o.amount
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE o.amount >= 60
ORDER BY
    c.customer_id,
    o.order_id;

customer_id,customer_name,order_id,amount
1,Ada,102,60
2,Bola,103,80


Chidi and Dami are gone. The left join kept them, and then `WHERE` ran on the joined rows and removed them, because a missing amount doesn't satisfy `>= 60` (a comparison with `NULL` is never true). The query quietly became an inner join.

To keep every customer while limiting which orders count as a match, move the amount condition into `ON`:

In [13]:
%%sql
SELECT
    c.customer_id,
    c.customer_name,
    o.order_id,
    o.amount
FROM customers AS c
LEFT JOIN orders AS o
    ON c.customer_id = o.customer_id
   AND o.amount >= 60
ORDER BY
    c.customer_id,
    o.order_id;

customer_id,customer_name,order_id,amount
1,Ada,102,60
2,Bola,103,80
3,Chidi,NULL,NULL
4,Dami,NULL,NULL


Neither placement is always right, because they answer different questions: `WHERE` asks "which joined rows do I keep?", while `ON` asks "what counts as a match?". A customer whose only orders were below 60 would get one unmatched row under the second query, and no row at all under the first, so decide which question you're asking before deciding where the condition goes.

### Joining on columns that only look related

Any two integer columns can be compared, so this runs without complaint:

In [14]:
%%sql
SELECT
    c.customer_name,
    o.order_id
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.order_id;

customer_name,order_id


It returns nothing here, because no customer ID happens to equal an order ID. In other data, accidental equal numbers would produce convincing pairs that are completely wrong, which is worse than an empty result. Check what each column in `ON` actually means, not just that the names look similar.

## 9. Try it yourself

Write each query in a new cell before looking at the [solutions notebook](<solutions/04-joins-solutions.ipynb>). Predict the result first, then run it; the gap between your prediction and the actual result is where the learning happens.

1. **Customers with no orders.** Return only the IDs and names of customers who have never placed an order. (You should get two names, and you can say which two before running anything.)
2. **Big orders per customer, including zero.** For every customer, count their orders of at least 60, so that Chidi and Dami appear with `0`. You'll need sections 7 and 8 together.
3. **Stretch, on AdventureWorks.** `Production.Product` lists every product and `Sales.SalesOrderDetail` has one row per product on each order. How many products have never been sold? Use `%%tsql` and the setup in the next section.

## 10. Scale it up: the same traps in AdventureWorks

Four customers make the mechanics visible, and real data is where the same mistakes hide. This section runs the same checks against **AdventureWorks2022**, Microsoft's sample database for a fictional bike company, on SQL Server. Three things change from here on: tables live in schemas (`Sales.Customer`, `Person.Person`), the dialect is T-SQL, which uses `TOP 5` where SQLite uses `LIMIT 5`, and the results are big enough that you can't eyeball them, which is the whole point.

These cells need SQL Server running with AdventureWorks restored; the [Course Overview](<00-course-overview.ipynb>) walks through setup. In Colab they'll stop with a "no SQL Server connection" message, and the saved results below are there so you can still follow along.

### Who has never ordered?

`Sales.Customer` has one row per customer and `Sales.SalesOrderHeader` one row per order. Count what a left join produces before looking at any rows:

In [15]:
%%tsql
SELECT
    COUNT(*) AS joined_rows,
    COUNT(s.SalesOrderID) AS matched_orders,
    COUNT(DISTINCT c.CustomerID) AS customers
FROM Sales.Customer AS c
LEFT JOIN Sales.SalesOrderHeader AS s
    ON c.CustomerID = s.CustomerID;

joined_rows,matched_orders,customers
32166,31465,19820


32,166 joined rows, 31,465 of them real orders, so 701 rows are customers the left join kept with no order at all. That's the same arithmetic as Chidi and Dami, just big enough that you'd never find it by scrolling. Here are the first few:

In [16]:
%%tsql
SELECT TOP 5
    c.CustomerID,
    c.PersonID,
    c.StoreID
FROM Sales.Customer AS c
LEFT JOIN Sales.SalesOrderHeader AS s
    ON c.CustomerID = s.CustomerID
WHERE s.SalesOrderID IS NULL
ORDER BY
    c.CustomerID;

CustomerID,PersonID,StoreID
1,NULL,934
2,NULL,1028
3,NULL,642
4,NULL,932
5,NULL,1026


### The fan-out bug, with real money

Every order in `Sales.SalesOrderHeader` has a `SubTotal`, and every order has one or more lines in `Sales.SalesOrderDetail`. Add up the subtotals at order grain first:

In [17]:
%%tsql
SELECT SUM(SubTotal) AS order_subtotal
FROM Sales.SalesOrderHeader;

order_subtotal
109846381.4039


About 109.8 million. Now join to the order lines, perhaps because the report also needs product details, and sum the same column:

In [18]:
%%tsql
SELECT
    SUM(h.SubTotal) AS joined_subtotal,
    SUM(d.LineTotal) AS line_total
FROM Sales.SalesOrderHeader AS h
INNER JOIN Sales.SalesOrderDetail AS d
    ON h.SalesOrderID = d.SalesOrderID;

joined_subtotal,line_total
2596470456.6774,109846381.399888


`joined_subtotal` is nearly 2.6 billion, about 24 times the real figure, and it's Ada's credit limit all over again: each order's subtotal repeats once per line, and some orders have 72 lines. `line_total`, a value that genuinely lives at line grain, still adds up to the real revenue (the difference in the last few decimal places comes from each line being rounded separately). Nothing errored, and a number like 2.6 billion is plausible enough to land in a slide deck, which is what makes this the most expensive join mistake on the list.

### WHERE undoes the LEFT JOIN, at scale

"Every customer, with their orders since the start of 2014." Here's the condition in `WHERE`, then in `ON`:

In [19]:
%%tsql
SELECT COUNT(DISTINCT c.CustomerID) AS customers_kept
FROM Sales.Customer AS c
LEFT JOIN Sales.SalesOrderHeader AS s
    ON c.CustomerID = s.CustomerID
WHERE s.OrderDate >= '2014-01-01';

customers_kept
10354


In [20]:
%%tsql
SELECT COUNT(DISTINCT c.CustomerID) AS customers_kept
FROM Sales.Customer AS c
LEFT JOIN Sales.SalesOrderHeader AS s
    ON c.CustomerID = s.CustomerID
   AND s.OrderDate >= '2014-01-01';

customers_kept
19820


10,354 customers versus all 19,820. If the report was meant to show every customer, the first version silently dropped nearly half of them.

### When two wrong numbers agree

One last trap, and it's a subtle one. Customer names live in `Person.Person`, so a natural first query joins to it:

In [21]:
%%tsql
SELECT COUNT(*) AS customers_with_a_person
FROM Sales.Customer AS c
INNER JOIN Person.Person AS p
    ON c.PersonID = p.BusinessEntityID;

customers_with_a_person
19119


19,119, which is exactly 19,820 minus the 701 customers with no orders. It looks like confirmation, and it isn't: the 701 customers dropped here are store accounts whose `PersonID` is `NULL` (you saw some in the "never ordered" query), and in this sample database they happen to be the same 701 customers who never ordered. On different data the two groups would diverge and the "confirmation" would vanish. Two numbers agreeing is not evidence that either is right; knowing why each row is in your result is.

## Before you trust a join

- Say in one sentence what a result row represents before you add anything up.
- Count rows before and after the join; if the number grew, find out which table multiplied it.
- Use `LEFT JOIN` when the unmatched rows are part of the answer, and check that no `WHERE` condition on the right-hand table quietly removes them again.
- Count a non-null key from the right-hand table, not `COUNT(*)`, when "zero" is a valid answer.
- Never leave out `ON`.

## Go deeper

- **[Say NO to Venn Diagrams When Explaining JOINs](https://blog.jooq.org/say-no-to-venn-diagrams-when-explaining-joins/)**, Lukas Eder. Why the Venn diagrams in most join tutorials mislead, and the "joins produce pairs" model this notebook uses instead.
- **[Joins (SQL Server)](https://learn.microsoft.com/en-us/sql/relational-databases/performance/joins)**, Microsoft Learn. The official reference for every join type in T-SQL, plus how SQL Server physically executes them, for when you're curious what happens under the hood.
- **[PostgreSQL tutorial: Joins Between Tables](https://www.postgresql.org/docs/current/tutorial-join.html)**. A short, clearly written walk through the same ideas in another major engine, useful for seeing what carries over unchanged.
- **[SQLite SELECT](https://www.sqlite.org/lang_select.html)**. The reference for the engine behind every `%%sql` cell above, including exactly how joins handle unmatched rows.
- **Practice:** [SQLZoo: The JOIN operation](https://sqlzoo.net/wiki/The_JOIN_operation) and [PostgreSQL Exercises: Joins](https://pgexercises.com/questions/joins/) both check your answers in the browser, and [DataLemur's joins tutorial](https://datalemur.com/sql-tutorial/sql-joins-inner-outer-full-left-right) frames the same ideas as interview questions.

For interview questions on this topic with model answers, see the [study guide](<study-guides/04-joins.md>).

## What's Next

You can now combine two tables and reason about what a join does to your row count and your totals. Next, `05 - Subqueries to CTEs` turns intermediate results into named steps, which is how longer queries that mix joins and summaries stay readable.

---

[← 03 - Aggregation](<03-aggregation.ipynb>) · [Course Overview](<00-course-overview.ipynb>) · [05 - Subqueries to CTEs →](<05-subqueries-to-ctes.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).